# Tanzania Tourism Cost Prediction
## Comprehensive Analysis, Feature Engineering & Model Optimization

## 1. Data Loading & Initial Exploration

In [ ]:
# Import essential libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Set style for better visualizations
sns.set_style('darkgrid')
plt.rcParams['figure.figsize'] = (14, 6)

# Load the training data
df = pd.read_csv('data/Train.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Detailed data information
print("Dataset Info:")
print(df.info())
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nMissing percentages:\n{(df.isnull().sum()/len(df)*100).round(2)}%")

## 2. Exploratory Data Analysis (EDA)

In [ ]:
# Statistical summary of numerical features
print("Numerical features statistics:")
print(df.describe())

# Check for outliers in target variable
plt.figure(figsize=(14, 4))

plt.subplot(1, 3, 1)
plt.hist(df['total_cost'], bins=50, edgecolor='black')
plt.title('Distribution of Total Cost')
plt.xlabel('Total Cost (TZS)')
plt.ylabel('Frequency')

plt.subplot(1, 3, 2)
plt.hist(np.log1p(df['total_cost']), bins=50, edgecolor='black')
plt.title('Distribution of Log-Transformed Total Cost')
plt.xlabel('Log(Total Cost)')
plt.ylabel('Frequency')

plt.subplot(1, 3, 3)
plt.boxplot(df['total_cost'])
plt.title('Total Cost Boxplot')
plt.ylabel('Total Cost (TZS)')

plt.tight_layout()
plt.show()

# Detect outliers using IQR method
Q1 = df['total_cost'].quantile(0.25)
Q3 = df['total_cost'].quantile(0.75)
IQR = Q3 - Q1
outliers = df[(df['total_cost'] < Q1 - 1.5*IQR) | (df['total_cost'] > Q3 + 1.5*IQR)]
print(f"\nOutliers detected: {len(outliers)} ({len(outliers)/len(df)*100:.2f}%)")

In [ ]:
# Categorical features analysis
categorical_cols = df.select_dtypes(include=['object']).columns
print(f"Categorical columns: {list(categorical_cols)}\n")

for col in categorical_cols:
    print(f"\n{col}:")
    print(f"  Unique values: {df[col].nunique()}")
    print(f"  Value counts:\n{df[col].value_counts()}")
    print(f"  Missing: {df[col].isnull().sum()}")

In [ ]:
# Correlation analysis with target variable
numeric_df = df.select_dtypes(include=[np.number])
correlation = numeric_df.corr()['total_cost'].sort_values(ascending=False)
print("Correlation with total_cost:")
print(correlation)

# Visualize correlations
plt.figure(figsize=(12, 8))
corr_matrix = numeric_df.corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix of Numerical Features')
plt.tight_layout()
plt.show()

In [ ]:
# Numerical features with high correlation
high_corr_features = correlation[abs(correlation) > 0.3].index.tolist()
print(f"Features with correlation > 0.3: {high_corr_features}")

# Visualize relationships
plt.figure(figsize=(15, 10))
for idx, col in enumerate(high_corr_features[1:7], 1):  # Skip target (first one)
    plt.subplot(2, 3, idx)
    plt.scatter(df[col], df['total_cost'], alpha=0.5)
    plt.xlabel(col)
    plt.ylabel('Total Cost')
    plt.title(f'{col} vs Total Cost')
plt.tight_layout()
plt.show()

In [ ]:
# Categorical features relationship with target
plt.figure(figsize=(15, 10))
cat_features = ['country', 'age_group', 'travel_with', 'purpose', 'main_activity']

for idx, col in enumerate(cat_features, 1):
    plt.subplot(2, 3, idx)
    df.groupby(col)['total_cost'].mean().sort_values().plot(kind='barh')
    plt.xlabel('Average Total Cost')
    plt.title(f'Average Cost by {col}')
plt.tight_layout()
plt.show()

## 3. Data Preprocessing & Cleaning

In [ ]:
# Create a copy for preprocessing
df_processed = df.copy()

# Handle missing values strategically
print("Handling missing values...")
df_processed['travel_with'] = df_processed['travel_with'].fillna('Alone')
df_processed['total_female'] = df_processed['total_female'].fillna(0).astype(int)
df_processed['total_male'] = df_processed['total_male'].fillna(0).astype(int)
df_processed['most_impressing'] = df_processed['most_impressing'].fillna('No comments')

# Standardize string columns
string_cols = df_processed.select_dtypes(include=['object']).columns
for col in string_cols:
    df_processed[col] = df_processed[col].str.strip()

# Convert yes/no to binary
binary_cols = ['first_trip_tz', 'package_transport_int', 'package_accomodation', 
                'package_food', 'package_transport_tz', 'package_sightseeing', 
                'package_guided_tour', 'package_insurance']

for col in binary_cols:
    if col in df_processed.columns:
        df_processed[col] = df_processed[col].str.lower().map({'yes': 1, 'no': 0})

print("\nMissing values after cleaning:")
print(df_processed.isnull().sum())

# Drop ID column
if 'ID' in df_processed.columns:
    df_processed = df_processed.drop('ID', axis=1)

print("\nData shape after cleaning:", df_processed.shape)

## 4. Advanced Feature Engineering

In [ ]:
# Create interaction and derived features
print("Creating advanced features...")

# 4.1 Group Size Features
df_processed['total_people'] = df_processed['total_female'] + df_processed['total_male']
df_processed['female_ratio'] = df_processed['total_female'] / (df_processed['total_people'] + 1)
df_processed['male_ratio'] = df_processed['total_male'] / (df_processed['total_people'] + 1)

# 4.2 Duration Features
df_processed['total_nights'] = df_processed['night_mainland'] + df_processed['night_zanzibar']
df_processed['mainland_ratio'] = df_processed['night_mainland'] / (df_processed['total_nights'] + 1)
df_processed['zanzibar_ratio'] = df_processed['night_zanzibar'] / (df_processed['total_nights'] + 1)

# 4.3 Package Features
package_cols = ['package_transport_int', 'package_accomodation', 'package_food', 
                 'package_transport_tz', 'package_sightseeing', 'package_guided_tour', 'package_insurance']
df_processed['package_count'] = df_processed[package_cols].sum(axis=1)
df_processed['package_ratio'] = df_processed['package_count'] / len(package_cols)

# 4.4 Interaction Features
df_processed['people_nights_interaction'] = df_processed['total_people'] * df_processed['total_nights']
df_processed['package_nights_interaction'] = df_processed['package_count'] * df_processed['total_nights']
df_processed['people_package_interaction'] = df_processed['total_people'] * df_processed['package_count']

# 4.5 Categorical grouping features
df_processed['purpose_grouped'] = df_processed['purpose'].replace({
    'Leisure and Holidays': 'Leisure',
    'Visiting Friends and Relatives': 'Leisure',
    'Business': 'Work',
    'Meetings and Conference': 'Work',
    'Scientific and Academic': 'Work',
    'Volunteering': 'Other',
    'Other': 'Other'
})

df_processed['activity_grouped'] = df_processed['main_activity'].replace({
    'Wildlife tourism': 'Wildlife',
    'Beach tourism': 'Beach',
    'Hunting tourism': 'Adventure',
    'Mountain climbing': 'Adventure',
    'Cultural tourism': 'Cultural',
    'Conference tourism': 'Work',
    'business': 'Work',
    'Bird watching': 'Other',
    'Diving and Sport Fishing': 'Other'
})

df_processed['impression_grouped'] = df_processed['most_impressing'].replace({
    'Friendly People': 'People',
    'No comments': 'NoComments',
    'Wildlife': 'Wildlife',
    'Wonderful Country, Landscape, Nature': 'Nature',
    'Good service': 'Other',
    'Excellent Experience': 'Other',
    'Satisfies and Hope Come Back': 'Other'
})

print("\nNew features created:")
print(df_processed[['total_people', 'total_nights', 'package_count', 
                     'people_nights_interaction']].head())
print(f"\nTotal features: {len(df_processed.columns)}")

In [ ]:
# Check multicollinearity (VIF - Variance Inflation Factor)
from statsmodels.stats.outliers_influence import variance_inflation_factor

numeric_features = df_processed.select_dtypes(include=[np.number]).columns
X_temp = df_processed[numeric_features].drop('total_cost', axis=1)

vif_data = pd.DataFrame()
vif_data['Feature'] = X_temp.columns
vif_data['VIF'] = [variance_inflation_factor(X_temp.values, i) for i in range(X_temp.shape[1])]
vif_data = vif_data.sort_values('VIF', ascending=False)

print("\nVariance Inflation Factor (VIF):")
print(vif_data[vif_data['VIF'] > 5])  # Remove features with VIF > 5

## 5. Feature Selection

In [ ]:
from sklearn.feature_selection import mutual_info_regression, SelectKBest, f_regression
from sklearn.preprocessing import LabelEncoder

# Prepare data for feature selection
df_encoded = df_processed.copy()

# Encode categorical variables
le_dict = {}
categorical_features = df_encoded.select_dtypes(include=['object']).columns

for col in categorical_features:
    le = LabelEncoder()
    df_encoded[col] = le.fit_transform(df_encoded[col])
    le_dict[col] = le

# Separate features and target
X = df_encoded.drop('total_cost', axis=1)
y = df_encoded['total_cost']

# Feature importance using mutual information
mi_scores = mutual_info_regression(X, y, random_state=42)
mi_scores_df = pd.DataFrame({
    'Feature': X.columns,
    'MI Score': mi_scores
}).sort_values('MI Score', ascending=False)

print("Top 15 Features by Mutual Information:")
print(mi_scores_df.head(15))

# Visualize
plt.figure(figsize=(12, 6))
plt.barh(mi_scores_df['Feature'][:15], mi_scores_df['MI Score'][:15])
plt.xlabel('Mutual Information Score')
plt.title('Top 15 Features by Mutual Information')
plt.tight_layout()
plt.show()

In [ ]:
# Select top features
selector = SelectKBest(score_func=f_regression, k=30)
X_selected = selector.fit_transform(X, y)
selected_features = X.columns[selector.get_support()].tolist()

print(f"Selected {len(selected_features)} features:")
print(selected_features)

## 6. Model Training & Evaluation

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import time

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {X_train.shape}")
print(f"Test set size: {X_test.shape}")

# Identify numerical and categorical columns
numerical_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"\nNumerical features: {len(numerical_features)}")
print(f"Categorical features: {len(categorical_features)}")

In [ ]:
# Create preprocessing pipeline
numerical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessor created successfully!")

In [ ]:
# Model configurations
models = {
    'Linear Regression': Pipeline([('preprocessor', preprocessor),
                                   ('model', LinearRegression())]),
    
    'Ridge (α=1.0)': Pipeline([('preprocessor', preprocessor),
                               ('model', Ridge(alpha=1.0, random_state=42))]),
    
    'Lasso (α=0.1)': Pipeline([('preprocessor', preprocessor),
                               ('model', Lasso(alpha=0.1, random_state=42))]),
    
    'ElasticNet': Pipeline([('preprocessor', preprocessor),
                            ('model', ElasticNet(alpha=0.1, random_state=42))]),
    
    'Random Forest': Pipeline([('preprocessor', preprocessor),
                               ('model', RandomForestRegressor(n_estimators=200, max_depth=15,
                                                               min_samples_split=5, min_samples_leaf=2,
                                                               random_state=42, n_jobs=-1))]),
    
    'Gradient Boosting': Pipeline([('preprocessor', preprocessor),
                                   ('model', GradientBoostingRegressor(n_estimators=200, max_depth=5,
                                                                       learning_rate=0.05, random_state=42))]),
    
    'AdaBoost': Pipeline([('preprocessor', preprocessor),
                          ('model', AdaBoostRegressor(n_estimators=100, learning_rate=0.1, random_state=42))]),
    
    'SVR (RBF)': Pipeline([('preprocessor', preprocessor),
                           ('model', SVR(kernel='rbf', C=100, gamma='scale'))])
}

print(f"Models to train: {list(models.keys())}")

In [ ]:
# Train and evaluate models
results = []

for name, model in models.items():
    print(f"\n{'='*50}")
    print(f"Training {name}...")
    print(f"{'='*50}")
    
    start_time = time.time()
    
    # Train model
    model.fit(X_train, y_train)
    
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)
    
    # Metrics
    train_r2 = r2_score(y_train, y_train_pred)
    test_r2 = r2_score(y_test, y_test_pred)
    train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
    test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
    train_mae = mean_absolute_error(y_train, y_train_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    
    # Cross-validation
    cv_scores = cross_val_score(model, X_train, y_train, cv=5, scoring='r2')
    
    elapsed_time = time.time() - start_time
    
    print(f"Train R²: {train_r2:.4f}")
    print(f"Test R²:  {test_r2:.4f}")
    print(f"Train RMSE: {train_rmse:.2f}")
    print(f"Test RMSE:  {test_rmse:.2f}")
    print(f"Train MAE: {train_mae:.2f}")
    print(f"Test MAE:  {test_mae:.2f}")
    print(f"CV R² (mean): {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
    print(f"Training time: {elapsed_time:.2f} seconds")
    
    results.append({
        'Model': name,
        'Train R²': train_r2,
        'Test R²': test_r2,
        'Train RMSE': train_rmse,
        'Test RMSE': test_rmse,
        'Train MAE': train_mae,
        'Test MAE': test_mae,
        'CV R² (mean)': cv_scores.mean(),
        'CV R² (std)': cv_scores.std(),
        'Time (s)': elapsed_time
    })

In [ ]:
# Results summary
results_df = pd.DataFrame(results)
print("\n" + "="*80)
print("MODEL COMPARISON SUMMARY")
print("="*80)
print(results_df.to_string())

# Best models by different metrics
print("\n" + "="*80)
print("BEST MODELS BY METRIC")
print("="*80)
print(f"Best Test R²: {results_df.loc[results_df['Test R²'].idxmax()]}")
print(f"\nBest Test RMSE: {results_df.loc[results_df['Test RMSE'].idxmin()]}")
print(f"\nBest Test MAE: {results_df.loc[results_df['Test MAE'].idxmin()]}")
print(f"\nBest CV R²: {results_df.loc[results_df['CV R² (mean)'].idxmax()]}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Test R² comparison
ax1 = axes[0, 0]
results_df_sorted = results_df.sort_values('Test R²', ascending=True)
ax1.barh(results_df_sorted['Model'], results_df_sorted['Test R²'], color='steelblue')
ax1.set_xlabel('Test R² Score')
ax1.set_title('Model Comparison: Test R²')
ax1.axvline(x=0, color='red', linestyle='--', alpha=0.5)

# Test RMSE comparison
ax2 = axes[0, 1]
results_df_sorted_rmse = results_df.sort_values('Test RMSE', ascending=False)
ax2.barh(results_df_sorted_rmse['Model'], results_df_sorted_rmse['Test RMSE'], color='coral')
ax2.set_xlabel('Test RMSE')
ax2.set_title('Model Comparison: Test RMSE (lower is better)')

# Train vs Test R² (overfitting check)
ax3 = axes[1, 0]
x = np.arange(len(results_df))
width = 0.35
ax3.bar(x - width/2, results_df['Train R²'], width, label='Train R²', alpha=0.8)
ax3.bar(x + width/2, results_df['Test R²'], width, label='Test R²', alpha=0.8)
ax3.set_ylabel('R² Score')
ax3.set_title('Train vs Test R² (Overfitting Check)')
ax3.set_xticks(x)
ax3.set_xticklabels(results_df['Model'], rotation=45, ha='right')
ax3.legend()
ax3.axhline(y=0, color='red', linestyle='--', alpha=0.5)

# Cross-validation performance
ax4 = axes[1, 1]
results_df_sorted_cv = results_df.sort_values('CV R² (mean)', ascending=True)
ax4.barh(results_df_sorted_cv['Model'], results_df_sorted_cv['CV R² (mean)'], color='lightgreen')
ax4.set_xlabel('CV R² Score (mean)')
ax4.set_title('Model Comparison: Cross-Validation R²')
ax4.axvline(x=0, color='red', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()